# Notebook 04 — Comparison & Alerting

The payoff notebook. Compares real portfolio and financial data against the correctly-resolved governing thresholds from Notebook 03, across multiple reporting periods per facility — proving the three claims this whole project was built around:

1. **Facility A** — the control case, healthy and compliant throughout.
2. **Facility B** — the same raw delinquency data, run through two different definitions, produces two very different risk pictures. Its Reserve Account covenant, still pending review, is correctly shown as unavailable rather than silently tested against an unapproved threshold.
3. **Facility C** — the resolved amendment produces the correct compliance picture; a naive system reading only the original agreement would have wrongly flagged two genuinely compliant periods as breaches.

**Input:** `records/03_record_store.json`, `synthetic_financials/*.csv`
**Output:** `records/04_comparison_output.json`

**No API calls in this notebook.**

In [1]:
import json
from datetime import date
from pathlib import Path

import pandas as pd

RECORDS_DIR = Path("../records")
FINANCIALS_DIR = Path("../synthetic_financials")

with open(RECORDS_DIR / "03_record_store.json") as f:
    record_store = json.load(f)


def deserialize_timeline(entries):
    result = []
    for e in entries:
        result.append({
            "value": e["value"],
            "from": date.fromisoformat(e["from"]) if e["from"] else None,
            "to": date.fromisoformat(e["to"]) if e["to"] else None,
            "priority": e["priority"],
            "source": e["source"],
        })
    return result


def resolve(timeline, as_of):
    for entry in timeline:
        after_start = entry["from"] is None or as_of >= entry["from"]
        before_end = entry["to"] is None or as_of <= entry["to"]
        if after_start and before_end:
            return entry
    return None


def get_governing_value(facility, covenant, as_of):
    """Returns the governing threshold string for a covenant at a given date,
    or None if the covenant has no approved record (e.g. still pending review)."""
    covenants = record_store.get(facility, {})
    if covenant not in covenants:
        return None
    timeline = deserialize_timeline(covenants[covenant])
    result = resolve(timeline, as_of)
    return result["value"] if result else None


df_a = pd.read_csv(FINANCIALS_DIR / "facility_a_periods.csv")
df_b = pd.read_csv(FINANCIALS_DIR / "facility_b_periods.csv")
df_c = pd.read_csv(FINANCIALS_DIR / "facility_c_periods.csv")

print("Record store loaded:", list(record_store.keys()))
print(f"Facility A: {len(df_a)} periods loaded")
print(f"Facility B: {len(df_b)} periods loaded")
print(f"Facility C: {len(df_c)} periods loaded")

Record store loaded: ['Facility A', 'Facility B', 'Facility C']
Facility A: 8 periods loaded
Facility B: 6 periods loaded
Facility C: 8 periods loaded


## Facility A — the control

Delinquency Ratio (60+ days ÷ current outstanding balance) and the Overcollateralization Test, computed period by period and compared against Facility A's own governing thresholds. This is the clean case — no drama expected, and that's the point: it proves the pipeline works correctly before we stress it with B and C.

In [2]:
threshold_a_delinq = float(get_governing_value("Facility A", "Delinquency Ratio / Delinquency Trigger", date(2024, 6, 30)).rstrip('%'))
threshold_a_oc = float(get_governing_value("Facility A", "Overcollateralization Test", date(2024, 6, 30)).rstrip('%'))

print(f"Governing Delinquency Trigger threshold: {threshold_a_delinq}%")
print(f"Governing Overcollateralization target: {threshold_a_oc}%\n")

facility_a_results = []
for _, row in df_a.iterrows():
    delinq_60plus = row["bucket_61_90"] + row["bucket_91_120"] + row["bucket_120_plus"]
    delinq_ratio = delinq_60plus / row["outstanding_pool_balance_eligible"] * 100
    triggered = delinq_ratio > threshold_a_delinq

    oc_amount = row["outstanding_pool_balance_eligible"] - row["advance_balance"]
    oc_target = (threshold_a_oc / 100) * row["outstanding_pool_balance_eligible"]
    oc_satisfied = oc_amount >= oc_target

    facility_a_results.append({
        "period": row["period"], "delinquency_ratio_pct": round(delinq_ratio, 3),
        "delinquency_trigger": triggered, "oc_amount": oc_amount,
        "oc_target": oc_target, "oc_satisfied": oc_satisfied,
    })

    print(f"  {row['period']}: DelinqRatio={delinq_ratio:.3f}%  ({'TRIGGER' if triggered else 'ok'})"
          f"   OC={'satisfied' if oc_satisfied else 'FAILED'} (amount={oc_amount:,.0f} vs target={oc_target:,.0f})")

Governing Delinquency Trigger threshold: 5.0%
Governing Overcollateralization target: 8.0%

  2024-03: DelinqRatio=0.800%  (ok)   OC=satisfied (amount=18,000,000 vs target=12,000,000)
  2024-04: DelinqRatio=0.929%  (ok)   OC=satisfied (amount=18,720,000 vs target=12,480,000)
  2024-05: DelinqRatio=1.056%  (ok)   OC=satisfied (amount=19,320,000 vs target=12,880,000)
  2024-06: DelinqRatio=1.159%  (ok)   OC=satisfied (amount=19,680,000 vs target=13,120,000)
  2024-07: DelinqRatio=1.257%  (ok)   OC=satisfied (amount=20,040,000 vs target=13,360,000)
  2024-08: DelinqRatio=1.361%  (ok)   OC=satisfied (amount=20,280,000 vs target=13,520,000)
  2024-09: DelinqRatio=1.462%  (ok)   OC=satisfied (amount=20,520,000 vs target=13,680,000)
  2024-10: DelinqRatio=1.536%  (ok)   OC=satisfied (amount=20,700,000 vs target=13,800,000)


## Facility B — definitional inconsistency, made concrete

The same raw delinquency data, computed two ways: correctly, using Facility B's own contractual definition (90+ days ÷ fixed original pool balance), and incorrectly, using Facility A's definition (60+ days ÷ current, shrinking balance) — the exact mistake someone would make reusing a familiar template without checking the governing definition.

Also checks Facility B's Reserve Account covenant — currently unresolved, still pending human review. Rather than silently skip it or guess a threshold, the correct behavior is to explicitly report it as untestable.

In [3]:
threshold_b_delinq = float(get_governing_value("Facility B", "Delinquency Ratio / Delinquency Trigger", date(2025, 1, 31)).rstrip('%'))
print(f"Governing Delinquency Trigger threshold (Facility B's own definition): {threshold_b_delinq}%\n")

facility_b_results = []
for _, row in df_b.iterrows():
    b_90plus = row["bucket_91_120"] + row["bucket_120_plus"]
    b_correct_ratio = b_90plus / row["original_pool_balance"] * 100
    b_triggered = b_correct_ratio > threshold_b_delinq

    a_60plus = row["bucket_61_90"] + row["bucket_91_120"] + row["bucket_120_plus"]
    a_wrong_ratio = a_60plus / row["outstanding_pool_balance_eligible"] * 100

    facility_b_results.append({
        "period": row["period"],
        "correct_ratio_pct": round(b_correct_ratio, 3), "correct_triggered": b_triggered,
        "if_wrongly_using_facility_a_definition_pct": round(a_wrong_ratio, 3),
    })

    print(f"  {row['period']}: B's own definition (90+/original) = {b_correct_ratio:.3f}%  "
          f"({'TRIGGER' if b_triggered else 'ok'} vs {threshold_b_delinq}%)"
          f"   |   if A's definition wrongly applied = {a_wrong_ratio:.3f}%")

reserve_threshold = get_governing_value("Facility B", "Reserve Account Deficiency / Required Reserve Amount", date(2025, 1, 31))
print(f"\nReserve Account Covenant: governing threshold = {reserve_threshold!r}")
if reserve_threshold is None:
    print("  -> Cannot be tested. This covenant is still pending human review (see Notebook 02's review queue)")
    print("     and has no approved governing threshold in the record store. Correctly excluded from monitoring")
    print("     rather than tested against an unverified value.")

Governing Delinquency Trigger threshold (Facility B's own definition): 4.0%

  2024-11: B's own definition (90+/original) = 0.273%  (ok vs 4.0%)   |   if A's definition wrongly applied = 0.688%
  2024-12: B's own definition (90+/original) = 0.441%  (ok vs 4.0%)   |   if A's definition wrongly applied = 1.123%
  2025-01: B's own definition (90+/original) = 0.714%  (ok vs 4.0%)   |   if A's definition wrongly applied = 1.848%
  2025-02: B's own definition (90+/original) = 1.159%  (ok vs 4.0%)   |   if A's definition wrongly applied = 3.020%
  2025-03: B's own definition (90+/original) = 1.841%  (ok vs 4.0%)   |   if A's definition wrongly applied = 4.868%
  2025-04: B's own definition (90+/original) = 2.795%  (ok vs 4.0%)   |   if A's definition wrongly applied = 7.623%

Reserve Account Covenant: governing threshold = None
  -> Cannot be tested. This covenant is still pending human review (see Notebook 02's review queue)
     and has no approved governing threshold in the record store. C

## Facility C — amendment resolution

Computes the Consolidated Net Leverage Ratio each Test Period, correctly resolving whichever threshold actually governs that specific date — 3.50:1.00 through Dec 2024, 4.00:1.00 from Mar 2025 onward — including the cure mechanic at Dec 2024. Then, for the two periods after the amendment takes effect, shows exactly what a naive system that only ever read the original agreement would have concluded instead.

In [4]:
def ratio_to_float(s):
    return float(s.split(':')[0])


facility_c_results = []
for _, row in df_c.iterrows():
    test_date = date.fromisoformat(row["test_period_end"])
    raw_ratio = row["consolidated_net_debt"] / row["consolidated_ebitda_ltm"]
    adj_ratio = row["consolidated_net_debt"] / (row["consolidated_ebitda_ltm"] + row["cure_contribution"])

    governing_str = get_governing_value("Facility C", "Consolidated Net Leverage Ratio", test_date)
    governing = ratio_to_float(governing_str)
    compliant = adj_ratio <= governing
    headroom = governing - adj_ratio

    facility_c_results.append({
        "test_period_end": row["test_period_end"], "raw_ratio": round(raw_ratio, 3),
        "adjusted_ratio": round(adj_ratio, 3), "governing_threshold": governing,
        "compliant": compliant, "headroom": round(headroom, 3),
    })

    cure_note = f"  (cure of {row['cure_contribution']:,.0f} applied)" if row["cure_contribution"] > 0 else ""
    print(f"  {row['test_period_end']}: raw={raw_ratio:.3f}x  adjusted={adj_ratio:.3f}x{cure_note}"
          f"  vs governing={governing:.2f}x  -> {'compliant' if compliant else 'BREACH'}  headroom={headroom:+.3f}x")

print("\nWhat a naive resolver (reads only the original agreement, never learns of the amendment) would conclude:")
naive_timeline = [{"value": "3.50:1.00", "from": date(2023, 9, 30), "to": None, "priority": 0}]
for test_date_str in ["2025-03-31", "2025-06-30"]:
    row = df_c[df_c["test_period_end"] == test_date_str].iloc[0]
    adj_ratio = row["consolidated_net_debt"] / (row["consolidated_ebitda_ltm"] + row["cure_contribution"])
    naive_compliant = adj_ratio <= 3.50
    print(f"  {test_date_str}: naive governing=3.50x  actual ratio={adj_ratio:.3f}x  "
          f"-> {'compliant' if naive_compliant else 'WRONGLY FLAGGED AS BREACH'}")

  2023-09-30: raw=2.760x  adjusted=2.760x  vs governing=3.50x  -> compliant  headroom=+0.740x
  2023-12-31: raw=2.898x  adjusted=2.898x  vs governing=3.50x  -> compliant  headroom=+0.602x
  2024-03-31: raw=3.065x  adjusted=3.065x  vs governing=3.50x  -> compliant  headroom=+0.435x
  2024-06-30: raw=3.240x  adjusted=3.240x  vs governing=3.50x  -> compliant  headroom=+0.260x
  2024-09-30: raw=3.424x  adjusted=3.424x  vs governing=3.50x  -> compliant  headroom=+0.076x
  2024-12-31: raw=3.667x  adjusted=3.483x  (cure of 3,000,000 applied)  vs governing=3.50x  -> compliant  headroom=+0.017x
  2025-03-31: raw=3.720x  adjusted=3.720x  vs governing=4.00x  -> compliant  headroom=+0.280x
  2025-06-30: raw=3.610x  adjusted=3.610x  vs governing=4.00x  -> compliant  headroom=+0.390x

What a naive resolver (reads only the original agreement, never learns of the amendment) would conclude:
  2025-03-31: naive governing=3.50x  actual ratio=3.720x  -> WRONGLY FLAGGED AS BREACH
  2025-06-30: naive govern

## Consolidated summary and save

Combines all three facilities' results into one saved output. Values coming out of pandas are numpy types, not plain Python types, which `json.dumps()` can't serialize directly — caught and fixed here with a small recursive converter, rather than discovered as a failure at save time.

In [5]:
def to_native(obj):
    """Recursively converts numpy scalar types to plain Python types so json.dumps works."""
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_native(v) for v in obj]
    if hasattr(obj, "item"):
        return obj.item()
    return obj


naive_comparison = {}
for test_date_str in ["2025-03-31", "2025-06-30"]:
    row = df_c[df_c["test_period_end"] == test_date_str].iloc[0]
    adj_ratio = row["consolidated_net_debt"] / (row["consolidated_ebitda_ltm"] + row["cure_contribution"])
    naive_comparison[test_date_str] = {
        "naive_threshold": 3.50,
        "actual_ratio": round(adj_ratio, 3),
        "naive_result": "WRONGLY FLAGGED AS BREACH" if adj_ratio > 3.50 else "compliant",
        "correct_result": "compliant",
    }

comparison_output = {
    "Facility A": {"covenant_tests": facility_a_results},
    "Facility B": {
        "delinquency_ratio_tests": facility_b_results,
        "reserve_account_covenant": {
            "status": "untestable - pending human review",
            "governing_threshold": reserve_threshold,
        },
    },
    "Facility C": {
        "leverage_ratio_tests": facility_c_results,
        "naive_resolver_comparison": naive_comparison,
    },
}

comparison_output = to_native(comparison_output)

output_path = RECORDS_DIR / "04_comparison_output.json"
output_path.write_text(json.dumps(comparison_output, indent=2))
print(f"Saved to {output_path}\n")

print("=== Overall summary ===")
print(f"Facility A: {sum(1 for r in facility_a_results if r['delinquency_trigger'])} delinquency trigger(s), "
      f"{sum(1 for r in facility_a_results if not r['oc_satisfied'])} OC failure(s) across {len(facility_a_results)} periods")
print(f"Facility B: {sum(1 for r in facility_b_results if r['correct_triggered'])} delinquency trigger(s) under correct definition "
      f"across {len(facility_b_results)} periods; Reserve Account untestable, pending review")
print(f"Facility C: {sum(1 for r in facility_c_results if not r['compliant'])} breach(es) across {len(facility_c_results)} periods "
      f"(correctly resolved); a naive resolver would have shown 2 false breaches instead")

Saved to ../records/04_comparison_output.json

=== Overall summary ===
Facility A: 0 delinquency trigger(s), 0 OC failure(s) across 8 periods
Facility B: 0 delinquency trigger(s) under correct definition across 6 periods; Reserve Account untestable, pending review
Facility C: 0 breach(es) across 8 periods (correctly resolved); a naive resolver would have shown 2 false breaches instead


## Summary

Reads back the saved comparison output from disk and presents the final, complete picture across all three facilities — the actual deliverable this whole project was built to produce.

In [6]:
with open(RECORDS_DIR / "04_comparison_output.json") as f:
    saved = json.load(f)

print("=" * 60)
print("FACILITY A -- control case")
print("=" * 60)
for r in saved["Facility A"]["covenant_tests"]:
    print(f"  {r['period']}: Delinquency={r['delinquency_ratio_pct']}%  OC satisfied={r['oc_satisfied']}")

print("\n" + "=" * 60)
print("FACILITY B -- definitional inconsistency")
print("=" * 60)
for r in saved["Facility B"]["delinquency_ratio_tests"]:
    print(f"  {r['period']}: correct={r['correct_ratio_pct']}%   "
          f"if A's definition wrongly applied={r['if_wrongly_using_facility_a_definition_pct']}%")
reserve = saved["Facility B"]["reserve_account_covenant"]
print(f"  Reserve Account: {reserve['status']}")

print("\n" + "=" * 60)
print("FACILITY C -- amendment resolution")
print("=" * 60)
for r in saved["Facility C"]["leverage_ratio_tests"]:
    print(f"  {r['test_period_end']}: {r['adjusted_ratio']}x vs governing {r['governing_threshold']}x  "
          f"-> {'compliant' if r['compliant'] else 'BREACH'}  headroom={r['headroom']:+.3f}x")
print("\n  If a naive resolver had been used instead (ignoring the amendment):")
for test_date, comparison in saved["Facility C"]["naive_resolver_comparison"].items():
    print(f"    {test_date}: {comparison['naive_result']}")

FACILITY A -- control case
  2024-03: Delinquency=0.8%  OC satisfied=True
  2024-04: Delinquency=0.929%  OC satisfied=True
  2024-05: Delinquency=1.056%  OC satisfied=True
  2024-06: Delinquency=1.159%  OC satisfied=True
  2024-07: Delinquency=1.257%  OC satisfied=True
  2024-08: Delinquency=1.361%  OC satisfied=True
  2024-09: Delinquency=1.462%  OC satisfied=True
  2024-10: Delinquency=1.536%  OC satisfied=True

FACILITY B -- definitional inconsistency
  2024-11: correct=0.273%   if A's definition wrongly applied=0.688%
  2024-12: correct=0.441%   if A's definition wrongly applied=1.123%
  2025-01: correct=0.714%   if A's definition wrongly applied=1.848%
  2025-02: correct=1.159%   if A's definition wrongly applied=3.02%
  2025-03: correct=1.841%   if A's definition wrongly applied=4.868%
  2025-04: correct=2.795%   if A's definition wrongly applied=7.623%
  Reserve Account: untestable - pending human review

FACILITY C -- amendment resolution
  2023-09-30: 2.76x vs governing 3.5x  